# Como Reparticionar e Otimizar Jobs em Spark

In [ ]:
from pyspark.sql import SparkSession

# Initialize SparkSession if not already active
if 'spark' not in locals():
    spark = SparkSession.builder.appName("RepartitioningExample").getOrCreate()

# Dados mal particionados (apenas 2 partições)
# Substituindo a leitura do arquivo por um DataFrame de exemplo, pois 'dados.csv' não foi encontrado.
# Criamos um DataFrame com 1000 linhas e uma coluna 'user_id' para simular os dados.
sample_data = [(i, f"user_{i%10}", i % 5) for i in range(1000)]
df = spark.createDataFrame(sample_data, ["id", "user_id", "value"]).repartition(2)

# Reparticionar para 200 partições (cluster com 50 cores)
df_otimizado = df.repartition(200)

# Ou particionar por coluna (útil para JOINs)
df_otimizado = df.repartition(200, "user_id")

In [ ]:
# --- Célula 1: Job Não Otimizado ---
from pyspark.sql.functions import col
import time

# Gerar dados grandes
df_vendas = spark.range(0, 1000000000).select(
    (col("id") % 1000).alias("produto_id"),
    (col("id") % 100).alias("valor")
)

df_produtos = spark.range(0, 1000).select(
    col("id").alias("produto_id"),
    col("id").alias("categoria")
)

# JOIN sem otimização
start = time.time()
resultado = df_vendas.join(df_produtos, "produto_id").groupBy("categoria").sum("valor")
resultado.count() # Action
end = time.time()

end_baseline = end
start_baseline = start
print(f"Tempo sem otimização: {end - start:.2f} segundos")

Tempo sem otimização: 34.71 segundos


In [6]:
# --- Célula 2: Job Otimizado ---
from pyspark.sql.functions import broadcast

# 1. Reparticionar
df_vendas_opt = df_vendas.repartition(200, "produto_id")

# 2. Broadcast da tabela pequena
start = time.time()
resultado_opt = df_vendas_opt.join(broadcast(df_produtos), "produto_id") \
    .groupBy("categoria").sum("valor")

# 3. Cache do resultado intermediário
resultado_opt.cache()
resultado_opt.count()
end = time.time()

print(f"Tempo otimizado: {end - start:.2f} segundos")
if (end - start) > 0:
    print(f"Speedup: {(end_baseline - start_baseline) / (end - start):.1f}x")

Tempo otimizado: 516.52 segundos
Speedup: 0.1x


### ⚠️ Por que o Speedup foi menor que 1x?

Existem três razões principais para a versão 'otimizada' ter demorado mais neste exemplo:

1. **Overhead de Agendamento:** Para um dataset pequeno (1.000 a 10.000.000 de linhas em memória), o custo de gerenciar 200 partições é maior do que o tempo de processamento real. O Spark gasta mais tempo 'conversando' com as tasks do que executando o cálculo.
2. **Custo do Repartition:** O comando `.repartition()` força um **Shuffle**, que é a operação mais cara do Spark (move dados pela rede). Em datasets pequenos, o custo do shuffle não compensa o ganho de paralelismo.
3. **Custo do Cache:** A primeira vez que você chama `.cache()`, o Spark precisa escrever os dados na memória (ou disco). O ganho do cache só aparece na *segunda* vez que você utiliza aquele DataFrame.

**Dica:** Reparticionamento e Broadcast brilham quando lidamos com **Terabytes** de dados, onde o gargalo é o processamento e não a coordenação do cluster.